In [1]:
# powershell
# uv add beautifulsoup4 requests

import requests
from bs4 import BeautifulSoup

In [3]:
url = "http://127.0.0.1:5500/notebooks/html/sample.html"

In [12]:
res = requests.get(url) # 페이지 요청 -> 응답을 받았다
#res.content
res.text[:200]

'<!DOCTYPE html>\n<html lang="ko">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>스크래핑 연습 상점</title>\n    <style>\n        body {\n '

In [25]:
soup = BeautifulSoup(res.text, 'html.parser') # DOM
soup.select("ul")

[<ul class="tags">
 <li>Python</li>
 <li>BeautifulSoup</li>
 </ul>,
 <ul class="tags">
 <li>Requests</li>
 <li>CSS Selector</li>
 </ul>,
 <ul class="tags">
 <li>Keyboard</li>
 </ul>]

In [28]:
# html 기준 요소를 찾는다
ul_tag = soup.find("ul", class_='tags')
li_tag = ul_tag.find_all("li")
li_tag

[<li>Python</li>, <li>BeautifulSoup</li>]

In [32]:
notice_table = soup.find("section", id="notices")
rows = notice_table.find("tbody").find_all("tr")

notices = []

for row in rows:
    cells = row.find_all("td")
    title_tag = row.find("a")
    time_tag = row.find("time")

    notices.append({
        "id": row.get("data-notice-id"),
        "number": cells[0].get_text(strip=True),
        "title": title_tag.get_text(strip=True),
        "url": title_tag.get("href"),
        "date": time_tag.get_text(strip=True),
        "datetime": time_tag.get("datetime"),
    })

for notice in notices:
    print(notice)

{'id': 'N003', 'number': '3', 'title': '추석 배송 일정 안내', 'url': '/notices/N003', 'date': '2026.09.15', 'datetime': '2026-09-15'}
{'id': 'N002', 'number': '2', 'title': '신규 강의 출시', 'url': '/notices/N002', 'date': '2026.09.10', 'datetime': '2026-09-10'}
{'id': 'N001', 'number': '1', 'title': '사이트 이용 안내', 'url': '/notices/N001', 'date': '2026.09.01', 'datetime': '2026-09-01'}


In [33]:
# powershell
# uv add pandas
import pandas as pd

notices_df = pd.DataFrame(notices)
notices_df

,id,number,title,url,date,datetime
0,N003,3,추석 배송 일정 안내,/notices/N003,2026.09.15,2026-09-15
1,N002,2,신규 강의 출시,/notices/N002,2026.09.10,2026-09-10
2,N001,1,사이트 이용 안내,/notices/N001,2026.09.01,2026-09-01


In [37]:
# html 페이지 수집 > 원하는 요소 > 추출 > 데이터프레임으로 가공 > 원하는 컬럼만 가져오기
notices_df['title']

0    추석 배송 일정 안내
1       신규 강의 출시
2      사이트 이용 안내
Name: title, dtype: str

In [45]:
naver_url = "https://stock.naver.com/market/marketindex"

res = requests.get(naver_url)

In [46]:
res.status_code, res.text[:200]

(200,
 '<!DOCTYPE html><html lang="ko" class="" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=1366, initial-scale=1.0, user-scalable=yes"/><meta name="viewport" content=')

In [47]:
soup = BeautifulSoup(res.text, 'html.parser') # DOM

In [49]:
url = "https://finance.naver.com/marketindex/exchangeList.naver"
m_index = requests.get(url)
soup = BeautifulSoup(m_index.content, 'html.parser')

table = soup.find('table', class_='tbl_exchange')
usd_row = table.find('tbody').find('tr')
us_price = usd_row.find(class_='sale')
print(us_price.text)


1,382.20


# 야후 파이낸스 환율 api 호출하기

In [50]:
# uv add yfinance

import requests
import yfinance as yf
from bs4 import BeautifulSoup

currency = "USD"
tickers = {"USD": "KRW=X", "JPY": "JPYKRW=X", "EUR": "EURKRW=X"}
ticker = tickers.get(currency)

if not ticker:
    print(f"{currency}는 지원하지 않는 통화입니다.")

price = yf.Ticker(ticker).info.get("regularMarketPrice")
price

1382.08

In [51]:
yf.Ticker(ticker).info

{'maxAge': 86400,
 'priceHint': 4,
 'previousClose': 1377.39,
 'open': 1377.46,
 'dayLow': 1373.18,
 'dayHigh': 1384.38,
 'regularMarketPreviousClose': 1377.39,
 'regularMarketOpen': 1377.46,
 'regularMarketDayLow': 1373.18,
 'regularMarketDayHigh': 1384.38,
 'volume': 0,
 'regularMarketVolume': 0,
 'averageVolume': 0,
 'averageVolume10days': 0,
 'averageDailyVolume10Day': 0,
 'bid': 1342.4,
 'ask': 1343.0,
 'bidSize': 0,
 'askSize': 0,
 'fiftyTwoWeekLow': 1322.42,
 'fiftyTwoWeekHigh': 1587.7,
 'allTimeHigh': 21353.0,
 'allTimeLow': 872.8,
 'fulldayPrice': 1382.18,
 'fulldayChange': 4.790039,
 'fulldayChangePercent': 0.34775698,
 'fiftyDayAverage': 1416.5793,
 'twoHundredDayAverage': 1465.5856,
 'currency': 'KRW',
 'tradeable': False,
 'quoteType': 'CURRENCY',
 'symbol': 'KRW=X',
 'language': 'en-US',
 'region': 'US',
 'typeDisp': 'Currency',
 'quoteSourceName': 'Delayed Quote',
 'triggerable': True,
 'customPriceAlertConfidence': 'HIGH',
 'regularMarketChange': 4.790039,
 'regularMark

# Yes24

In [77]:
# https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=24

In [2]:
class Book:
    def __init__(self, rank, title, author, price):
        self.rank = rank
        self.title = title
        self.author = author
        self.price = price


    def __str__(self):
        return f"{self.rank}, {self.title}, {self.author}, {self.price}"
   
    def to_dict(self):
        return {'rank':self.rank,
                'title':self.title,
                'author':self.author,
                'price':self.price}
   
    def to_list(self):
        return [self.rank,
                self.title,
                self.author,
                self.price]


In [4]:
# 데이터를 사전에 준비할 때

page_no = 3
book_list = []
rank = 0

for page in range(1, page_no+1):
    yes_url = f"https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001&pageNumber={page}"
    res = requests.get(yes_url)
    soup = BeautifulSoup(res.text, "html.parser")

    best_list_el = soup.select('#yesBestList div.item_info')
    for item in best_list_el:
        rank += 1
        title = item.select_one('div.info_name > a').text
        # author = item.select_one('div.info_pubGrp > span.info_auth > a').text 
        author = item.select_one('.info_auth > a').text 
        price = int(item.select_one('div.info_price .txt_num').text.replace(",","").replace("원",""))
        book_list.append(Book(rank, title, author, price))

In [5]:
len(book_list)

72

In [6]:
book_list[0].to_list()

[1, '세네카, 오늘을 빼앗기고 있는 당신에게', '루키우스 안나이우스 세네카', 16200]

1. 데이터베이스 생성
2. 테이블 생성 create table
3. 데이터 삽입 insert
4. 데이터 검색 select

In [7]:
sql = """
CREATE TABLE IF NOT EXISTS Books (
    rank    INTEGER PRIMARY KEY,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    price   INTEGER NOT NULL CHECK (price >= 0)
);
"""

In [8]:
# 데이터베이스 구성
import sqlite3
conn = sqlite3.connect('my_database.db') #1 데이터베이스 생성
cursor = conn.cursor()
cursor.execute(sql) #2 테이블생성

In [9]:
ins_sql = """
INSERT INTO Books (rank, title, author, price) VALUES (?, ?, ?, ?)
"""

In [10]:
# 데이터 적재
for book in book_list:
    cursor.execute(ins_sql, book.to_list())

In [11]:
conn.commit()

In [12]:
conn.close()